# Capstone: Refresh / Content Opportunity Scoring (Lane 2)

**Question:** using only what is known at the end of March 2026, which content pages are most at risk of losing more than 20% of their daily impressions in April 2026, so they should be reviewed for a refresh first?

- Dev month (features): `2026-03` - Label month: `2026-04`
- The sealed test month `2026-06` is **never** touched.
- Everything is hashed in the release: no client names, URLs or queries appear anywhere.
- Run the cells top to bottom. Before running: add your Hugging Face token as a Colab Secret named `HF_TOKEN` (key icon, left side) and turn on Notebook access.

In [ ]:
!pip -q install duckdb huggingface_hub scikit-learn matplotlib pandas pyarrow

In [ ]:
import json, os, re, warnings
import numpy as np, pandas as pd, duckdb
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

REPO = "FlyRank/internship-warehouse"
DEV_MONTH, LABEL_MONTH = "2026-03", "2026-04"     # 2026-06 (sealed test) is never used
DROP_THRESHOLD = 0.20      # label = daily impressions fall by more than 20%
MIN_IMPR = 100             # ignore pages with fewer than 100 impressions in March (too noisy)
RANDOM_STATE = 42
os.makedirs("results", exist_ok=True)
print("token found:", bool(HF_TOKEN))

## 1. Connect to the warehouse and list files

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
all_files = [f for f in api.list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)
             if f.endswith(".parquet")]
PREFIX = f"hf://datasets/{REPO}/"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")
print(len(all_files), "parquet files in the repo")

In [ ]:
def urls_for(table, month=None):
    """Parquet URLs of one table (optionally one month). The sealed month/sample tables are always excluded."""
    fs = [f for f in all_files if table in f.split("/")]
    if not fs:
        fs = [f for f in all_files if table in f]
    if month:
        fs = [f for f in fs if f"month={month}" in f]
    fs = [f for f in fs if "_sample" not in f and "2026-06" not in f]
    return [PREFIX + f for f in fs]

def rp(urls):
    return "read_parquet([" + ", ".join(f"'{u}'" for u in urls) + "], union_by_name=true)"

def q(sql):
    return con.execute(sql).fetchdf()

fact_mar = urls_for("fact_content_daily_performance", DEV_MONTH)
fact_apr = urls_for("fact_content_daily_performance", LABEL_MONTH)
dim_c    = urls_for("dim_content")
print("March files:", len(fact_mar), "| April files:", len(fact_apr), "| dim_content files:", len(dim_c))
assert fact_mar and fact_apr and dim_c, "Could not find the tables - send me the output of the cell above."

## 2. Find the real column names (so nothing is hardcoded blindly)

In [ ]:
fact_cols = list(q(f"DESCRIBE SELECT * FROM {rp(fact_mar[:1])}").column_name)
dim_cols  = list(q(f"DESCRIBE SELECT * FROM {rp(dim_c[:1])}").column_name)
print("fact columns:", fact_cols)
print("dim_content columns:", dim_cols)

def pick(names, patterns, exclude=()):
    for p in patterns:
        for n in names:
            if re.fullmatch(p, n) and not any(re.search(e, n) for e in exclude):
                return n
    return None

C = {
    "date":    pick(fact_cols, [r"report_date", r".*date.*"]),
    "client":  pick(fact_cols, [r"client_hash_id"]),
    "content": pick(fact_cols, [r"content_hash_id"]),
    "impr":    pick(fact_cols, [r"gsc_impressions", r"impressions", r".*impressions.*"], exclude=["ai", "ctr", "avg", "rate"]),
    "clicks":  pick(fact_cols, [r"gsc_clicks", r"clicks", r".*clicks.*"], exclude=["ai", "ctr", "rate"]),
    "pos":     pick(fact_cols, [r"gsc_avg_position", r"avg_position", r"gsc_position", r"position", r".*avg.*position.*"], exclude=["sum"]),
}
D = {
    "content": pick(dim_cols, [r"content_hash_id"]),
    "created": pick(dim_cols, [r"content_created_at", r".*created.*"]),
}
print("\nColumn mapping used:", C, D)
missing = [k for k in ["date", "client", "content", "impr", "clicks"] if C[k] is None] + \
          [k for k in ["content"] if D[k] is None]
assert not missing, f"Could not detect: {missing}. Send me the two column lists printed above."

## 3. Data contract (plain words)

- **Grain:** one row per (client, content page), summarised over the 31 days of March 2026.
- **Tables:** `fact_content_daily_performance` (March features, April label) + `dim_content` (page age).
- **Decision moment:** end of March 2026. Every feature uses data up to 2026-03-31 only.
- **Target:** 1 if the page's daily impressions in April are more than 20% lower than its March daily impressions.
- **Population:** pages with at least 100 impressions in March, from clients that have April data (so a missing April row means a real drop, not a client that left the data).
- **Excluded:** product decision outputs (health/priority scores, action flags) and anything from April as a feature.

## 4. Build features (March) and label (April)

In [ ]:
H = lambda col: f'"{col}"'
pos_expr = f'CAST({H(C["pos"])} AS DOUBLE)' if C["pos"] else "CAST(NULL AS DOUBLE)"

feat_sql = f"""
WITH d AS (
  SELECT {H(C["client"])} AS client, {H(C["content"])} AS content,
         CAST({H(C["date"])} AS DATE) AS d,
         CAST({H(C["impr"])} AS DOUBLE) AS impr,
         CAST({H(C["clicks"])} AS DOUBLE) AS clicks,
         {pos_expr} AS pos
  FROM {rp(fact_mar)}
)
SELECT client, content,
  SUM(impr) AS impr_30d,
  SUM(clicks) AS clicks_30d,
  SUM(pos * impr) / NULLIF(SUM(CASE WHEN pos IS NOT NULL THEN impr END), 0) AS pos_w,
  COUNT(DISTINCT d) AS active_days,
  SUM(CASE WHEN d <  DATE '{DEV_MONTH}-16' THEN impr   ELSE 0 END) AS impr_h1,
  SUM(CASE WHEN d >= DATE '{DEV_MONTH}-16' THEN impr   ELSE 0 END) AS impr_h2,
  SUM(CASE WHEN d <  DATE '{DEV_MONTH}-16' THEN clicks ELSE 0 END) AS clicks_h1,
  SUM(CASE WHEN d >= DATE '{DEV_MONTH}-16' THEN clicks ELSE 0 END) AS clicks_h2
FROM d
GROUP BY 1, 2
HAVING SUM(impr) >= {MIN_IMPR}
"""
feat = q(feat_sql)
print("March pages:", len(feat), "| clients:", feat.client.nunique())

lab = q(f"""
SELECT {H(C["client"])} AS client, {H(C["content"])} AS content,
       SUM(CAST({H(C["impr"])} AS DOUBLE)) AS impr_next
FROM {rp(fact_apr)}
GROUP BY 1, 2
""")
april_clients = set(lab.client.unique())
print("April rows:", len(lab), "| April clients:", len(april_clients))

if D["created"]:
    age_src = q(f'SELECT {H(D["content"])} AS content, TRY_CAST({H(D["created"])} AS TIMESTAMP) AS created FROM {rp(dim_c)}')
else:
    age_src = pd.DataFrame({"content": [], "created": []})

In [ ]:
df = feat[feat.client.isin(april_clients)].merge(lab, on=["client", "content"], how="left")
df["impr_next"] = df["impr_next"].fillna(0.0)          # present in April's client data, but no rows -> 0 impressions

# per-day rates (March has 31 days, April has 30)
daily_mar = df.impr_30d / 31.0
daily_apr = df.impr_next / 30.0
df["ratio_next"] = daily_apr / daily_mar
df["y"] = (df["ratio_next"] < (1 - DROP_THRESHOLD)).astype(int)

# ---- features: all of them use data up to 2026-03-31 only ----
df["log_impr"]   = np.log1p(df.impr_30d)
df["log_clicks"] = np.log1p(df.clicks_30d)
df["ctr_30d"]    = df.clicks_30d / df.impr_30d
df["momentum_impr"]   = np.log((df.impr_h2 / 16 + 1)   / (df.impr_h1 / 15 + 1))
df["momentum_clicks"] = np.log((df.clicks_h2 / 16 + 1) / (df.clicks_h1 / 15 + 1))
client_total = df.groupby("client")["impr_30d"].transform("sum")
df["client_share_log"] = np.log(df.impr_30d / client_total)
df = df.merge(age_src.drop_duplicates("content"), on="content", how="left")
df["content_age_days"] = (pd.Timestamp(f"{DEV_MONTH}-31") - pd.to_datetime(df["created"])).dt.days.clip(lower=0)

FEATURES = ["log_impr", "log_clicks", "ctr_30d", "pos_w", "active_days",
            "momentum_impr", "momentum_clicks", "client_share_log", "content_age_days"]
FEATURES = [f for f in FEATURES if df[f].notna().any()]
base_rate = float(df.y.mean())
print(f"rows: {len(df):,} | clients: {df.client.nunique()} | base rate of a >20% drop: {base_rate:.1%}")
print("features used:", FEATURES)
assert 0.02 < base_rate < 0.98, "Label is almost constant - send me this output."

### Why each feature is available at the decision moment (end of March)

| Feature | Available at decision time because... |
|---|---|
| `log_impr`, `log_clicks`, `ctr_30d` | summed over March 1-31 only |
| `pos_w` | impression-weighted average position during March |
| `active_days` | count of days with a row in March |
| `momentum_impr`, `momentum_clicks` | second half of March vs first half of March |
| `client_share_log` | page share of its client's March impressions |
| `content_age_days` | creation date known before the decision, measured at 2026-03-31 |

April data appears **only** in the label.

## 5. Verification queries (grain, row count / date span, availability)

In [ ]:
print("Grain check - total rows vs distinct (date, client, content):")
print(q(f"""
SELECT COUNT(*) AS total_rows,
       COUNT(DISTINCT ({H(C['date'])}, {H(C['client'])}, {H(C['content'])})) AS distinct_keys
FROM {rp(fact_mar)}""").to_string(index=False))

print("\nRow count and date span (March):")
print(q(f"""
SELECT COUNT(*) AS rows, MIN({H(C['date'])}) AS first_day, MAX({H(C['date'])}) AS last_day
FROM {rp(fact_mar)}""").to_string(index=False))

if "ga4_data_available" in fact_cols:
    print("\nAvailability flag (IS TRUE, so NULLs are not counted as available):")
    print(q(f"""
    SELECT COUNT(*) AS rows, SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {rp(fact_mar)}""").to_string(index=False))

## 6. Honest evaluation: split by client, baseline vs models

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
tr_idx, te_idx = next(gss.split(df, df.y, groups=df.client))
train, test = df.iloc[tr_idx], df.iloc[te_idx]
assert set(train.client).isdisjoint(set(test.client))          # no client appears on both sides
train = train.sample(min(len(train), 400_000), random_state=RANDOM_STATE)
test  = test.sample(min(len(test), 300_000), random_state=RANDOM_STATE)
assert test.y.nunique() == 2 and train.y.nunique() == 2, "Only one class in a split."
print(f"train: {len(train):,} rows / {train.client.nunique()} clients | test: {len(test):,} rows / {test.client.nunique()} clients")

def prec_at(y, score, frac=0.10):
    k = max(1, int(len(y) * frac))
    idx = np.argsort(-np.asarray(score))[:k]
    return float(np.asarray(y)[idx].mean())

def evaluate(name, score, y=test.y.values):
    auc = roc_auc_score(y, score)
    p = prec_at(y, score)
    print(f"{name:<34} AUC {auc:.3f} | precision@top10% {p:.3f} | lift {p / y.mean():.2f}x")
    return {"auc": float(auc), "precision_top10": p, "lift_top10": float(p / y.mean())}

res = {}
# baseline: rank pages by how much they were already falling inside March
res["baseline"] = evaluate("Baseline (March momentum only)", -test.momentum_impr.fillna(0).values)

logit = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
logit.fit(train[FEATURES], train.y)
res["logistic"] = evaluate("Logistic regression", logit.predict_proba(test[FEATURES])[:, 1])

hgb = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_depth=4, random_state=RANDOM_STATE)
hgb.fit(train[FEATURES], train.y)
test_score = hgb.predict_proba(test[FEATURES])[:, 1]
res["model"] = evaluate("Gradient boosting (final model)", test_score)

### Is the result stable? 5-fold cross-validation by client

The single split above has only a few test clients, so one lucky or unlucky split could mislead. Here every client is held out exactly once.

In [ ]:
from sklearn.model_selection import GroupKFold
rows = []
for k, (a, b) in enumerate(GroupKFold(n_splits=5).split(df, df.y, groups=df.client), 1):
    tr_, te_ = df.iloc[a], df.iloc[b]
    if te_.y.nunique() < 2:
        continue
    tr_ = tr_.sample(min(len(tr_), 400_000), random_state=RANDOM_STATE)
    m = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_depth=4, random_state=RANDOM_STATE)
    m.fit(tr_[FEATURES], tr_.y)
    rows.append({"fold": k, "test_clients": int(te_.client.nunique()), "test_pages": int(len(te_)),
                 "baseline_auc": roc_auc_score(te_.y, -te_.momentum_impr.fillna(0).values),
                 "model_auc": roc_auc_score(te_.y, m.predict_proba(te_[FEATURES])[:, 1])})
cv = pd.DataFrame(rows)
print(cv.round(3).to_string(index=False))
print(f"\nBaseline AUC: {cv.baseline_auc.mean():.3f} +/- {cv.baseline_auc.std():.3f}")
print(f"Model AUC:    {cv.model_auc.mean():.3f} +/- {cv.model_auc.std():.3f}")
res["cv_5fold_by_client"] = {
    "folds": int(len(cv)),
    "baseline_auc_mean": float(cv.baseline_auc.mean()), "baseline_auc_std": float(cv.baseline_auc.std()),
    "model_auc_mean": float(cv.model_auc.mean()), "model_auc_std": float(cv.model_auc.std()),
    "per_fold_model_auc": [float(x) for x in cv.model_auc],
}

## 7. Leakage trap: add a label-derived column on purpose, watch the score jump, then remove it

In [ ]:
LEAK = "leak_log_impr_next"            # built from April impressions = the answer itself
train_l, test_l = train.copy(), test.copy()
train_l[LEAK] = np.log1p(train_l.impr_next)
test_l[LEAK]  = np.log1p(test_l.impr_next)

leaky = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.08, max_depth=4, random_state=RANDOM_STATE)
leaky.fit(train_l[FEATURES + [LEAK]], train_l.y)
leaky_auc = float(roc_auc_score(test_l.y, leaky.predict_proba(test_l[FEATURES + [LEAK]])[:, 1]))
honest_auc = res["model"]["auc"]
print(f"WITH the leaked column:    AUC {leaky_auc:.3f}   <- looks amazing, and is meaningless")
print(f"WITHOUT it (honest model): AUC {honest_auc:.3f}   <- the number I keep")
res["leakage_demo"] = {"leaky_auc": leaky_auc, "honest_auc": honest_auc}
del train_l, test_l, leaky                  # the leaked column is deleted; nothing below uses it

## 8. What drives the score, and the ranked recommendations

In [ ]:
from sklearn.inspection import permutation_importance
samp = test.sample(min(len(test), 50_000), random_state=RANDOM_STATE)
pi = permutation_importance(hgb, samp[FEATURES], samp.y, scoring="roc_auc", n_repeats=3, random_state=RANDOM_STATE)
imp = pd.Series(pi.importances_mean, index=FEATURES).sort_values(ascending=False)
print(imp.round(4).to_string())
res["importance"] = {k: float(v) for k, v in imp.items()}

rec = test.copy()
rec["risk"] = test_score
rec = rec.sort_values("risk", ascending=False).reset_index(drop=True)
rec["rank"] = np.arange(1, len(rec) + 1)
rec["decile"] = pd.qcut(rec["rank"], 10, labels=False) + 1       # 1 = highest risk

ctr_q25 = rec.ctr_30d.quantile(0.25)
def reasons(r):
    out = []
    if r.momentum_impr < -0.20:  out.append("IMPRESSIONS_FALLING")
    if r.momentum_clicks < -0.20: out.append("CLICKS_FALLING")
    if r.ctr_30d <= ctr_q25:      out.append("LOW_CTR")
    if pd.notna(r.content_age_days) and r.content_age_days > 365: out.append("AGING_CONTENT")
    return "|".join(out) if out else "MONITOR"
top = rec[rec.decile == 1].copy()
top["reason_codes"] = top.apply(reasons, axis=1)

dec = rec.groupby("decile")["y"].mean()
print("\nActual drop rate by predicted-risk decile (1 = highest risk):")
print(dec.round(3).to_string())
res["decile_drop_rate"] = {int(k): float(v) for k, v in dec.items()}

codes = top.reason_codes.str.split("|").explode().value_counts()
res["top_decile_reason_codes"] = {k: int(v) for k, v in codes.items()}
res["top_decile_size"] = int(len(top))
print("\nReason codes in the top-risk decile:")
print(codes.to_string())

# public-safe export: no client or content ids
top[["rank", "risk", "momentum_impr", "ctr_30d", "pos_w", "content_age_days", "reason_codes"]] \
    .head(500).to_csv("results/top_recommendations.csv", index=False)

## 9. Charts

In [ ]:
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False})

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
names = ["Baseline", "Logistic", "Boosting"]
keys  = ["baseline", "logistic", "model"]
ax[0].bar(names, [res[k]["auc"] for k in keys], color=["#9aa5b1", "#7aa2d6", "#1f6feb"])
ax[0].axhline(0.5, color="k", ls="--", lw=0.8); ax[0].set_title("ROC AUC (held-out clients)"); ax[0].set_ylim(0.4, 1)
ax[1].bar(names, [res[k]["lift_top10"] for k in keys], color=["#9aa5b1", "#7aa2d6", "#1f6feb"])
ax[1].axhline(1, color="k", ls="--", lw=0.8); ax[1].set_title("Lift in the top 10% highest-risk pages")
plt.tight_layout(); plt.savefig("results/model_vs_baseline.png"); plt.show()

fig, ax = plt.subplots(figsize=(4.5, 3.6))
ax.bar(["With leaked\ncolumn", "Honest\nmodel"], [leaky_auc, honest_auc], color=["#d1453b", "#1f6feb"])
ax.set_ylim(0.4, 1.02); ax.set_title("Leakage trap: ROC AUC")
plt.tight_layout(); plt.savefig("results/leakage_trap.png"); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.8))
imp.sort_values().plot.barh(ax=ax, color="#1f6feb"); ax.set_title("Permutation importance (drop in AUC)")
plt.tight_layout(); plt.savefig("results/feature_importance.png"); plt.show()

fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(dec.index, dec.values, color="#1f6feb"); ax.axhline(base_rate, color="k", ls="--", lw=0.8)
ax.set_xlabel("Predicted-risk decile (1 = highest)"); ax.set_ylabel("Actual share with a >20% drop")
ax.set_title("Does a higher score really mean more drops?")
plt.tight_layout(); plt.savefig("results/decile_drop_rate.png"); plt.show()

## 10. Save the numbers (paste the printed JSON back to Claude to write the paper)

In [ ]:
summary = {
    "dev_month": DEV_MONTH, "label_month": LABEL_MONTH, "drop_threshold": DROP_THRESHOLD, "min_impressions": MIN_IMPR,
    "n_pages": int(len(df)), "n_clients": int(df.client.nunique()),
    "n_train_pages": int(len(train)), "n_test_pages": int(len(test)), "n_test_clients": int(test.client.nunique()),
    "base_rate_drop": base_rate, "features": FEATURES, "columns_used": {**C, **{"dim_" + k: v for k, v in D.items()}},
    **res,
}
with open("results/results.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 11. Limitations (honest framing)

- The label compares **one** month to the next month, so it can pick up seasonality rather than page decay; results are *observed / directional / decision-support*, not proof of what causes a drop.
- Only pages with 100+ March impressions are scored; very small pages are out of scope.
- A 20% fall in impressions is a reviewed-by-a-human trigger, not an automatic rewrite order.
- Clients with a later tracking start are under-represented in the March slice.
- The sealed month `2026-06` was not touched, so a final sealed-test check is still a separate step.